# SI section 6: Inferring the initial mutant location
Each cell prints one sentence of the section and every number the sentence quotes.
The column SI is the quoted value and the column computed is the value at the precision of the SI.
Data: `fixloc.sim.large` on a cluster for the simulated networks, `python -m fixloc.make.si.inference` for the graphs with $N=8$ vertices, and `fixloc.origin` for the graphs with $N=10$ vertices.

In [1]:
import json

import numpy as np

from fixloc.make.si import quote

load = lambda f: json.load(open(f))
pct = '{:.0%}'
from collections import Counter

from fixloc import origin

rows = load('large_summary.json')['rows']
mem = load('origin_memory_N8.json')

## Exact computation for small graphs

In [2]:
fam = Counter(f for f, _, _ in origin.families(10))
quote(
  r'The graphs with $N=10$ in \Cref{fig:origin} are the cycle, path, star, complete graph, and $2\times5$ grid, and five instances each of six random graph families.',
  ('35', sum(fam.values()), '{}'),
)
print(dict(fam))

The graphs with N=10 in [ref] are the cycle, path, star, complete graph, and 2 x 5 grid, and five instances each of six random graph families.
    SI         35   computed         35   (35)   ok

{'cycle': 1, 'path': 1, 'star': 1, 'complete': 1, 'grid': 1, 'random 3-regular': 5, 'Erdos-Renyi': 5, 'Barabasi-Albert': 5, 'small-world': 5, 'random tree': 5, 'random digraph': 5}


## Simulation for larger graphs

In [3]:
size = lambda n: 50 if n <= 50 else (100 if n <= 100 else 200)
nets = {x['id']: x for x in rows}
quote(
  r'The structures are a cycle, a periodic square grid, and three instances each of the six random graph families above.',
  *[(si, sum(1 for x in nets.values() if x['family'] == f and size(x['N']) == 200), '{}') for f, si in (('cycle', '1'), ('torus', '1'), ('Erdos-Renyi', '3'))],
)
print('networks by family and size:', dict(Counter((x['family'], size(x['N'])) for x in nets.values())))
quote(
  r'The grids are $7\times7$, $10\times10$, and $14\times14$.',
  ('49, 100, 196', sorted(x['N'] for x in nets.values() if x['family'] == 'torus'), lambda v: ', '.join(map(str, v))),
)
quote(
  r'The Erd\H{o}s--R\'enyi graphs have mean degree $4$.',
  ('4', np.mean([x['links'] / x['N'] for x in nets.values() if x['family'] == 'Erdos-Renyi']), '{:.0f}'),
)
quote(
  r'For each initial vertex we simulated $4{,}000$ runs ending in fixation.',
  ('4,000', sorted({x['takeovers'] // x['N'] for x in rows}), lambda v: ', '.join(f'{n:,}' for n in v)),
)

The structures are a cycle, a periodic square grid, and three instances each of the six random graph families above.
    SI          1   computed          1   (1)   ok
    SI          1   computed          1   (1)   ok
    SI          3   computed          3   (3)   ok

networks by family and size: {('random 3-regular', 50): 3, ('Erdos-Renyi', 50): 3, ('Barabasi-Albert', 50): 3, ('small-world', 50): 3, ('random tree', 50): 3, ('random directed', 50): 3, ('torus', 50): 1, ('cycle', 50): 1, ('random 3-regular', 100): 3, ('Erdos-Renyi', 100): 3, ('Barabasi-Albert', 100): 3, ('small-world', 100): 3, ('random tree', 100): 3, ('random directed', 100): 3, ('torus', 100): 1, ('cycle', 100): 1, ('random 3-regular', 200): 3, ('Erdos-Renyi', 200): 3, ('Barabasi-Albert', 200): 3, ('small-world', 200): 3, ('random tree', 200): 3, ('random directed', 200): 3, ('torus', 200): 1, ('cycle', 200): 1}
The grids are 7 x 7, 10 x 10, and 14 x 14.
    SI 49, 100, 196   computed 49, 100, 196   ([49, 100, 196]

In [4]:
g = {}
for x in rows:
  if x['r'] == 2.0:
    g.setdefault(x['family'], {}).setdefault(size(x['N']), []).append(x['acc_location'] / x['prior_best'])
mean = {f: {n: float(np.mean(v)) for n, v in d.items()} for f, d in g.items()}
quote(
  r'At $r=2$ the gain on the cycle is $3.22$, $4.40$, and $6.29$ for $N=50$, $100$, and $200$.',
  *[(si, mean['cycle'][n], '{:.2f}') for n, si in ((50, '3.22'), (100, '4.40'), (200, '6.29'))],
)
quote(
  r'On the periodic grid the gain is $1.28$, $1.60$, and $2.35$.',
  *[(si, mean['torus'][n], '{:.2f}') for n, si in ((50, '1.28'), (100, '1.60'), (200, '2.35'))],
)
quote(
  r'On the random trees the mean gain is between $1.65$ and $1.90$.',
  ('1.65', min(mean['random tree'].values()), '{:.2f}'),
  ('1.90', max(mean['random tree'].values()), '{:.2f}'),
)
grows = all(np.diff([mean[f][n] for n in (50, 100, 200)]).min() > 0 for f in ('cycle', 'torus'))
quote(r'The gain grows with $N$ on the cycle and on the periodic grid.', ('True', grows, '{}'))
well = ('Erdos-Renyi', 'Barabasi-Albert', 'small-world', 'random 3-regular', 'random directed')
quote(
  r'At $N=200$ the mean gain on the five well-connected families is between $0.97$ and $1.24$.',
  ('0.97', min(mean[f][200] for f in well), '{:.2f}'),
  ('1.24', max(mean[f][200] for f in well), '{:.2f}'),
)
print({f: {n: round(v, 2) for n, v in d.items()} for f, d in mean.items()})

At r=2 the gain on the cycle is 3.22, 4.40, and 6.29 for N=50, 100, and 200.
    SI       3.22   computed       3.22   (3.22035)   ok
    SI       4.40   computed       4.40   (4.39657)   ok
    SI       6.29   computed       6.29   (6.29448)   ok

On the periodic grid the gain is 1.28, 1.60, and 2.35.
    SI       1.28   computed       1.28   (1.28065)   ok
    SI       1.60   computed       1.60   (1.596)   ok
    SI       2.35   computed       2.35   (2.34609)   ok

On the random trees the mean gain is between 1.65 and 1.90.
    SI       1.65   computed       1.65   (1.65092)   ok
    SI       1.90   computed       1.90   (1.89525)   ok

The gain grows with N on the cycle and on the periodic grid.
    SI       True   computed       True   (True)   ok

At N=200 the mean gain on the five well-connected families is between 0.97 and 1.24.
    SI       0.97   computed       0.97   (0.965526)   ok
    SI       1.24   computed       1.24   (1.24209)   ok

{'random 3-regular': {50: 1.36, 10

## The population structures with information about the origin

In [5]:
quote(
  r'For every connected undirected graph with $N=8$ vertices we computed the mutual information between the initial vertex and the fixation location.',
  *[('11,117', mem[r]['n_graphs'], '{:,}') for r in mem],
)
for r, m in mem.items():
  print(f"r = {r}: mutual information from {m['MI_min']:.3f} to {m['MI_max']:.3f} bits, median {m['MI_median']:.3f}")
quote(r'The information is largest on elongated sparse graphs with cut vertices.')
for r, m in mem.items():
  print(f"r = {r}: graph of largest information has {m['max_graph']['links']} edges, diameter {m['max_graph']['diameter']}, degrees {m['max_graph']['degrees']}")
quote(r'The information is smallest on the complete graph and on the graphs obtained from the complete graph by removing one or two edges.')
for r, m in mem.items():
  print(f'r = {r}: rank by increasing information of the graphs with 0, 1, and 2 missing edges:', sorted((x['missing_links'], x['rank']) for x in m['near_complete']))
  print('        missing edges of the five graphs with the least information:', [x['missing_links'] for x in m['lowest']])

For every connected undirected graph with N=8 vertices we computed the mutual information between the initial vertex and the fixation location.
    SI     11,117   computed     11,117   (11117)   ok
    SI     11,117   computed     11,117   (11117)   ok
    SI     11,117   computed     11,117   (11117)   ok

r = 1.25: mutual information from 0.004 to 0.612 bits, median 0.078
r = 2.0: mutual information from 0.009 to 0.599 bits, median 0.116
r = 10.0: mutual information from 0.081 to 0.673 bits, median 0.238
The information is largest on elongated sparse graphs with cut vertices.

r = 1.25: graph of largest information has 7 edges, diameter 7, degrees [2, 2, 2, 2, 2, 2, 1, 1]
r = 2.0: graph of largest information has 9 edges, diameter 5, degrees [3, 3, 2, 2, 2, 2, 2, 2]
r = 10.0: graph of largest information has 11 edges, diameter 4, degrees [4, 3, 3, 3, 3, 2, 2, 2]
The information is smallest on the complete graph and on the graphs obtained from the complete graph by removing one or tw

In [6]:
sp = {r: m['spearman'] for r, m in mem.items()}
quote(
  r'The information increases with the diameter and with the mean distance between vertices.',
  *[('> 0', sp[r][k], lambda x: '> 0' if x > 0 else f'{x:.2f}') for r in sp for k in ('diameter', 'mean distance')],
)
quote(
  r'The information decreases with the number of edges and with the minimum degree.',
  *[('< 0', sp[r][k], lambda x: '< 0' if x < 0 else f'{x:.2f}') for r in sp for k in ('links', 'minimum degree')],
)
print('Spearman rank correlations with the mutual information:', {r: {k: round(v, 2) for k, v in s.items()} for r, s in sp.items()})

The information increases with the diameter and with the mean distance between vertices.
    SI        > 0   computed        > 0   (0.78171)   ok
    SI        > 0   computed        > 0   (0.784717)   ok
    SI        > 0   computed        > 0   (0.783165)   ok
    SI        > 0   computed        > 0   (0.806889)   ok
    SI        > 0   computed        > 0   (0.784069)   ok
    SI        > 0   computed        > 0   (0.85523)   ok

The information decreases with the number of edges and with the minimum degree.
    SI        < 0   computed        < 0   (-0.65879)   ok
    SI        < 0   computed        < 0   (-0.835945)   ok
    SI        < 0   computed        < 0   (-0.687548)   ok
    SI        < 0   computed        < 0   (-0.83185)   ok
    SI        < 0   computed        < 0   (-0.752748)   ok
    SI        < 0   computed        < 0   (-0.711287)   ok

Spearman rank correlations with the mutual information: {'1.25': {'diameter': 0.78, 'mean distance': 0.78, 'links': -0.66, 'minimum